# 02 · Experimentación — hiperparámetros y evaluación
**TT 2027-A190 · CRISP-DM fases 4 (modelado) y 5 (evaluación)**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_RAW, MODELS_DIR, TrainConfig
from src.grid_search import run_grid
from src.train import run

SUBSET = "FD001"
EPOCHS = 60          

## 1. Búsqueda en cuadrícula (ventana × celdas LSTM × abandono)

In [ ]:
base = TrainConfig(subset=SUBSET, epochs=EPOCHS)
WINDOWS = [20, 30]
LSTM_OPTIONS = [(32, 32), (64, 32)]
DROPOUTS = [0.1, 0.2]
grid = run_grid(base, WINDOWS, LSTM_OPTIONS, DROPOUTS, DATA_RAW)
grid

In [ ]:
best = grid.iloc[0]
print("Mejor configuración por RMSE de validación:\n", best)
fig, ax = plt.subplots(figsize=(8, 3.6))
for w, g in grid.groupby("window"):
    ax.plot(range(len(g)), g["val_rmse"].values, "o-", label=f"ventana {w}")
ax.set(xlabel="Configuración (ordenadas por val_rmse)", ylabel="RMSE de validación"); ax.legend(); plt.show()

## 2. Entrenamiento final y evaluación en prueba

In [ ]:
cfg = TrainConfig(subset=SUBSET, epochs=EPOCHS, window=int(best["window"]),
                  lstm_units=tuple(best["lstm_units"]), dropout=float(best["dropout"]))
metrics = run(cfg, DATA_RAW, MODELS_DIR, verbose=0)
pd.DataFrame({"prueba (RUL real sin tope)": metrics["test"],
              "prueba (RUL real topada)": metrics["test_capped_labels"],
              "validación (ventanas)": metrics["validation_windows"]}).T

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(MODELS_DIR / SUBSET / "figures" / "pred_vs_real.png")))

## 3. Comparativa FD001–FD004
Ejecute `make train-all` y consolide las métricas. Complete la tabla de literatura con los valores reportados en los artículos originales (Zheng et al., 2017; Li et al., 2018).

In [ ]:
rows = []
for s in ["FD001", "FD002", "FD003", "FD004"]:
    p = MODELS_DIR / s / "metrics.json"
    if p.exists():
        m = json.loads(p.read_text(encoding="utf-8"))["test"]
        rows.append({"subconjunto": s, "RMSE": m["rmse"], "MAE": m["mae"], "Score": m["score"]})
pd.DataFrame(rows).round(2)